In [8]:
import os
import json
from dotenv import dotenv_values
from IPython.display import display, Markdown

In [9]:
config = dotenv_values(".env")

endpoint = config.get("AZURE_DOC_ENDPOINT")
key = config.get("AZURE_DOC_KEY")
model_id = config.get("AZURE_DOC_DEPLOYMENT") or "prebuilt-layout"

print("Endpoint:", endpoint)
print("Model:", model_id)

Endpoint: https://doc-intel-nour.cognitiveservices.azure.com/
Model: prebuilt-layout


In [10]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentContentFormat,DocumentAnalysisFeature


client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

def analyze_document(file_path, selected_model=model_id):
    with open(file_path, "rb") as f:
        poller = client.begin_analyze_document(
            model_id=selected_model,
            body=f,
            content_type="application/octet-stream",
            features=[DocumentAnalysisFeature.key_value_pairs],
            output_content_format=DocumentContentFormat.MARKDOWN
        )
        result = poller.result()
    return result



In [11]:
images_dir = "images"
outputs_dir = "outputs"

os.makedirs(images_dir, exist_ok=True)
os.makedirs(outputs_dir, exist_ok=True)

supported_exts = ('.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.pdf')
image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(supported_exts)]

print("Found files:", len(image_files))
for f in image_files:
    print("-", f)

Found files: 8
- 10171108.jpeg
- WhatsApp Image 2026-10-15 at 11.58.49 PM.jpeg
- WhatsApp Image 2026-10-16 at 12.00.22 AM.jpeg
- WhatsApp Image 2026-10-16 at 12.00.48 AM.jpeg
- WhatsApp Image 2026-10-16 at 12.02.58 AM.jpeg
- WhatsApp Image 2026-10-16 at 12.03.33 AM.jpeg
- WhatsApp Image 2026-10-16 at 12.06.48 AM.jpeg
- WhatsApp Image 2026-10-16 at 12.07.25 AM.jpeg


In [12]:
filename="10171108.jpeg"

In [13]:
image_path = os.path.join(images_dir, filename)
base_name = os.path.splitext(filename)[0]

result = analyze_document(image_path)
result_dict = result.as_dict() if hasattr(result, "as_dict") else dict(result)

with open(os.path.join(outputs_dir, f"{base_name}.json"), "w", encoding="utf-8") as f:
    json.dump(result_dict, f, ensure_ascii=False, indent=2)
    
with open(os.path.join(outputs_dir, f"{base_name}.md"), "w", encoding="utf-8") as f:
    f.write(result.content if result.content else "")
    
print(f"Processed: {filename}")

Processed: 10171108.jpeg
